# Análise de Correspondência Múltipla (MCA)

**Tema:** perfil de consumo em plataformas de streaming.

Neste notebook, será aplicada a **Análise de Correspondência Múltipla (MCA)** para investigar associações entre variáveis categóricas de clientes fictícios, como faixa etária, dispositivo utilizado, conteúdo preferido, tipo de plano e período de uso.

A MCA é uma extensão da ANACOR para mais de duas variáveis categóricas. Enquanto a ANACOR normalmente utiliza uma matriz de contingência como entrada, a MCA utiliza o banco de dados original com as variáveis categóricas.

In [ ]:
#%% Instalação dos pacotes, caso necessário

# Descomente as linhas abaixo se algum pacote não estiver instalado no seu ambiente.
# !pip install pandas
# !pip install numpy
# !pip install scipy
# !pip install matplotlib
# !pip install prince

#%% Importando os pacotes necessários

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import prince

In [ ]:
#%% Importando o banco de dados

# Dataset fictício criado para fins didáticos e publicação no GitHub.
# Cada linha representa um cliente e suas características categóricas de consumo em streaming.

streaming_mca = pd.read_csv("clientes_streaming_mca.csv")

streaming_mca.head()

In [ ]:
#%% Selecionando apenas as variáveis categóricas que serão usadas na MCA

# A coluna cliente_id é apenas um identificador e não deve entrar na análise.

dados_mca = streaming_mca.drop(columns=["cliente_id"])

dados_mca.head()

In [ ]:
#%% Informações descritivas das variáveis categóricas

# Esta etapa ajuda a entender a distribuição das categorias antes da análise.

for coluna in dados_mca.columns:
    print(f"\nVariável: {coluna}")
    print(dados_mca[coluna].value_counts())

In [ ]:
#%% Analisando algumas tabelas de contingência

# Antes da MCA, é útil observar cruzamentos entre pares de variáveis.
# Essas tabelas ajudam a identificar possíveis associações preliminares.

tabela_1 = pd.crosstab(dados_mca["faixa_etaria"], dados_mca["dispositivo_principal"])
tabela_2 = pd.crosstab(dados_mca["faixa_etaria"], dados_mca["conteudo_preferido"])
tabela_3 = pd.crosstab(dados_mca["dispositivo_principal"], dados_mca["tipo_plano"])

print("Faixa etária x Dispositivo principal")
print(tabela_1)

print("\nFaixa etária x Conteúdo preferido")
print(tabela_2)

print("\nDispositivo principal x Tipo de plano")
print(tabela_3)

In [ ]:
#%% Teste qui-quadrado para verificar associação entre pares de variáveis

# O teste qui-quadrado avalia se duas variáveis categóricas são independentes.
# H0: as variáveis são independentes.
# H1: existe associação entre as variáveis.

def teste_quiquadrado(tabela, nome):
    chi2, p_valor, gl, esperados = chi2_contingency(tabela)
    print(f"\nAssociação: {nome}")
    print(f"Qui-quadrado: {chi2:.2f}")
    print(f"p-valor: {p_valor:.4f}")
    print(f"Graus de liberdade: {gl}")
    
    if p_valor < 0.05:
        print("Conclusão: rejeita-se H0. Há evidências de associação.")
    else:
        print("Conclusão: não se rejeita H0. Não há evidências fortes de associação.")

teste_quiquadrado(tabela_1, "Faixa etária x Dispositivo principal")
teste_quiquadrado(tabela_2, "Faixa etária x Conteúdo preferido")
teste_quiquadrado(tabela_3, "Dispositivo principal x Tipo de plano")

In [ ]:
#%% Elaborando a MCA

# Na MCA, o input é o próprio banco de dados com variáveis categóricas.
# Isso é diferente da ANACOR, em que normalmente usamos uma matriz de contingência.

mca = prince.MCA(
    n_components=2,
    random_state=42
).fit(dados_mca)

In [ ]:
#%% Quantidade total de dimensões possíveis na MCA

# Na MCA, a quantidade máxima de dimensões é dada por:
# quantidade total de categorias - quantidade de variáveis.

quantidade_categorias = mca.J_
quantidade_variaveis = mca.K_
quantidade_dimensoes = quantidade_categorias - quantidade_variaveis

print(f"Quantidade total de categorias: {quantidade_categorias}")
print(f"Quantidade de variáveis: {quantidade_variaveis}")
print(f"Quantidade máxima de dimensões: {quantidade_dimensoes}")

In [ ]:
#%% Visualizando a matriz binária e a matriz de Burt

# A MCA trabalha com uma representação binária das categorias.
# Cada categoria vira uma coluna indicadora, assumindo valor 1 quando o indivíduo pertence àquela categoria.
# A matriz de Burt é obtida pelo produto da matriz binária transposta pela matriz binária.

binaria = pd.get_dummies(dados_mca, columns=dados_mca.columns, dtype=float)
burt = np.matmul(binaria.T, binaria)

print("Matriz binária:")
print(binaria.head())

print("\nMatriz de Burt:")
print(pd.DataFrame(burt, index=binaria.columns, columns=binaria.columns).head())

In [ ]:
#%% Obtendo os autovalores e a inércia explicada

# Os autovalores indicam quanta informação cada dimensão explica.
# Na MCA e na ANACOR, essa informação é chamada de inércia.

tabela_autovalores = mca.eigenvalues_summary

print(tabela_autovalores)

In [ ]:
#%% Inércia principal total

# A inércia total representa a informação associativa total presente na análise.

print(f"Inércia total: {mca.total_inertia_:.4f}")

In [ ]:
#%% Coordenadas principais das categorias

# As coordenadas permitem transformar categorias em pontos numéricos no plano.
# Cada categoria recebe uma coordenada na Dimensão 1 e outra na Dimensão 2.

coord_categorias = mca.column_coordinates(dados_mca)

print(coord_categorias)

In [ ]:
#%% Coordenadas-padrão das categorias

# As coordenadas-padrão ajudam a visualizar melhor a posição relativa das categorias no mapa perceptual.

coord_padrao = mca.column_coordinates(dados_mca) / np.sqrt(mca.eigenvalues_)

print(coord_padrao)

In [ ]:
#%% Coordenadas das observações

# Cada cliente também pode ser representado como um ponto no espaço da MCA.
# Neste projeto, o foco será no mapa das categorias.

coord_observacoes = mca.row_coordinates(dados_mca)

print(coord_observacoes.head())

In [ ]:
#%% Preparando os dados para o mapa perceptual

chart = coord_padrao.reset_index()

# O prince cria nomes como variavel__categoria ou variavel_categoria, dependendo da versão.
# A função abaixo tenta separar o nome da variável de forma robusta.

def extrair_variavel(nome_categoria):
    nome_categoria = str(nome_categoria)
    for coluna in dados_mca.columns:
        if nome_categoria.startswith(coluna):
            return coluna
    return "categoria"

chart_df_mca = pd.DataFrame({
    "categoria": chart["index"],
    "x": chart[0],
    "y": chart[1],
})

chart_df_mca["variavel"] = chart_df_mca["categoria"].apply(extrair_variavel)

chart_df_mca

In [ ]:
#%% Plotando o mapa perceptual da MCA

# No mapa perceptual, categorias próximas tendem a estar mais associadas.
# O eixo X representa a 1ª dimensão, e o eixo Y representa a 2ª dimensão.

plt.figure(figsize=(10, 7), dpi=300)

variaveis = chart_df_mca["variavel"].unique()

for variavel in variaveis:
    temp = chart_df_mca[chart_df_mca["variavel"] == variavel]
    plt.scatter(temp["x"], temp["y"], label=variavel, s=45)
    
    for _, row in temp.iterrows():
        plt.text(row["x"] + 0.03, row["y"] + 0.02, row["categoria"], fontsize=7)

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.title("Mapa perceptual - MCA\nPerfil de Consumo em Streaming")
plt.xlabel("1ª dimensão (X)")
plt.ylabel("2ª dimensão (Y)")
plt.legend(title="Variável", fontsize=7)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Interpretação esperada

O mapa perceptual deve ser interpretado pela proximidade entre as categorias. Categorias próximas no gráfico tendem a apresentar maior associação no banco de dados.

Neste dataset fictício, espera-se observar padrões como:

- clientes mais jovens próximos de **Smartphone**, **Séries**, **Animes** e uso no período da **Noite/Madrugada**;
- clientes adultos próximos de **Smart TV**, **Filmes** e plano **Padrão**;
- clientes mais velhos próximos de **Documentários**, **Filmes clássicos**, **Premium** e uso no período da **Tarde/Fim de semana**.

Essas associações dependem das coordenadas finais calculadas pela MCA.

## Observação

Uma diferença prática importante entre ANACOR e MCA em Python é o formato dos dados de entrada.

Na **ANACOR**, o input costuma ser uma matriz de contingência, construída a partir do cruzamento entre duas variáveis categóricas.

Na **MCA**, o input costuma ser a base original com várias variáveis categóricas, sem necessidade de montar manualmente uma única tabela de contingência.